<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/06_topology_optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 6 of 7: Topology Optimisation

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

Can we rearrange voxels to minimise tortuosity at a fixed porosity? This tutorial uses OpenImpala as the cost-function evaluator inside two different optimisation strategies and compares their results.

**Why OpenImpala for optimisation?** Each iteration requires a full 3D PDE solve to evaluate the objective function. A typical optimisation campaign needs hundreds to thousands of evaluations. With a serial solver, this is painfully slow — a 100-iteration loop on a $64^3$ grid might take hours. OpenImpala's MPI-parallel solver makes each evaluation fast enough that meaningful optimisation becomes practical:

| Grid size | Evaluations | Serial solver | OpenImpala (32 cores) |
|-----------|------------|--------------|----------------------|
| $24^3$ | 100 | ~10 min | ~30 sec |
| $64^3$ | 500 | ~12 hours | ~20 min |
| $128^3$ | 1000 | days | ~3 hours |

On HPC, you can also run **population-based** optimisers (genetic algorithms, particle swarm) where each member of the population is evaluated independently — scaling linearly with cluster size via job arrays or MPI task farming.

Both algorithms below use the same mutation operator — swap a random solid voxel with a random pore voxel, so porosity stays exactly fixed — but differ in their acceptance rule:

- **Greedy hill-climbing**: accept a swap only if it improves (lowers) the tortuosity. Simple, but gets trapped in local optima.
- **Simulated annealing (SA)**: sometimes accept worsening swaps early on (controlled by a "temperature" that cools over time), allowing the search to escape local optima before converging.

By running both on the same starting structure, we can see the practical difference.

**What you will learn:**
1. Use OpenImpala as a physics evaluator inside an optimisation loop.
2. Implement greedy and simulated annealing acceptance rules.
3. Compare convergence behaviour and final structures.
4. Analyse what the optimiser found relative to the theoretical bound.

**Prerequisites:** [Tutorial 1](01_hello_openimpala.ipynb) for the core API. [Tutorial 5](05_surrogate_modelling.ipynb) introduced the idea of using OpenImpala in automated pipelines — here we close the loop with an optimiser.

In [ ]:
!pip install -q openimpala matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

SIZE = 24          # Domain side length (24^3 = 13,824 voxels)
N_ITERATIONS = 100 # Swap attempts per algorithm run
POROSITY = 0.50    # Fixed porosity for all experiments

np.random.seed(42)
initial_micro = np.random.choice([0, 1], size=(SIZE, SIZE, SIZE), p=[1-POROSITY, POROSITY]).astype(np.int32)
print(f"Initial structure: {SIZE}^3, porosity = {initial_micro.mean():.4f}")

In [ ]:
def propose_swap(micro):
    """Pick one random pore voxel and one random solid voxel."""
    pore_coords = np.argwhere(micro == 1)
    solid_coords = np.argwhere(micro == 0)
    p = pore_coords[np.random.choice(len(pore_coords))]
    s = solid_coords[np.random.choice(len(solid_coords))]
    return tuple(p), tuple(s)


def run_optimisation(micro_start, n_iter, method="greedy", T_init=0.15, cooling=0.97):
    """Run optimisation and return (final_micro, tau_history, best_history, n_accepted).

    method: "greedy" or "sa" (simulated annealing).
    """
    micro = micro_start.copy()
    best_micro = micro.copy()

    # Evaluate starting point
    perc = oi.percolation_check(micro, phase=1, direction="z")
    current_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity if perc.percolates else np.inf
    best_tau = current_tau

    tau_history = [current_tau]   # tracks current state (SA may go up)
    best_history = [best_tau]     # tracks best-so-far
    T = T_init
    accepted = 0

    for i in range(n_iter):
        p_idx, s_idx = propose_swap(micro)

        # Apply swap
        micro[p_idx] = 0
        micro[s_idx] = 1

        # Evaluate
        perc = oi.percolation_check(micro, phase=1, direction="z")
        new_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity if perc.percolates else np.inf

        # Acceptance rule
        delta = new_tau - current_tau
        if method == "greedy":
            accept = delta < 0
        else:  # simulated annealing
            accept = delta < 0 or (T > 0 and np.random.random() < np.exp(-delta / T))
            T *= cooling

        if accept:
            current_tau = new_tau
            accepted += 1
            if new_tau < best_tau:
                best_tau = new_tau
                best_micro = micro.copy()
        else:
            # Revert
            micro[p_idx] = 1
            micro[s_idx] = 0

        tau_history.append(current_tau)
        best_history.append(best_tau)

    return best_micro, tau_history, best_history, accepted

In [ ]:
print("Running both optimisation strategies on the same starting structure...\n")

with oi.Session():
    # --- Greedy hill-climbing ---
    t0 = time.time()
    greedy_micro, greedy_tau, greedy_best, greedy_acc = run_optimisation(
        initial_micro, N_ITERATIONS, method="greedy"
    )
    t_greedy = time.time() - t0
    print(f"Greedy:  tau {greedy_tau[0]:.4f} -> {greedy_best[-1]:.4f} "
          f"({greedy_acc}/{N_ITERATIONS} accepted, {t_greedy:.1f}s)")

    # --- Simulated annealing ---
    t0 = time.time()
    sa_micro, sa_tau, sa_best, sa_acc = run_optimisation(
        initial_micro, N_ITERATIONS, method="sa", T_init=0.15, cooling=0.97
    )
    t_sa = time.time() - t0
    print(f"SA:      tau {sa_tau[0]:.4f} -> {sa_best[-1]:.4f} "
          f"({sa_acc}/{N_ITERATIONS} accepted, {t_sa:.1f}s)")

## 1. Convergence Comparison

The plot below shows both the "current state" tortuosity (which can go up for SA) and the "best found so far" for each method. Notice that SA's current state wanders upward early on — it is intentionally exploring worse configurations — but its best-so-far can ultimately surpass the greedy result by escaping local optima.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=120)
iters = range(N_ITERATIONS + 1)

# Left: current state (SA can go up, greedy only goes down)
ax1.plot(iters, greedy_tau, '-', color='#d62728', lw=1.5, alpha=0.8, label='Greedy (current)')
ax1.plot(iters, sa_tau, '-', color='#1f77b4', lw=1.5, alpha=0.8, label='SA (current)')
ax1.set_xlabel("Iteration")
ax1.set_ylabel(r"Current Tortuosity ($\tau$)")
ax1.set_title("Current State")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: best-so-far (monotonically decreasing for both)
ax2.plot(iters, greedy_best, '-', color='#d62728', lw=2.5, label=f'Greedy (final: {greedy_best[-1]:.4f})')
ax2.plot(iters, sa_best, '-', color='#1f77b4', lw=2.5, label=f'SA (final: {sa_best[-1]:.4f})')
ax2.axhline(1.0, color='gray', ls=':', lw=1, label='Theoretical min (straight channel)')
ax2.set_xlabel("Iteration")
ax2.set_ylabel(r"Best Tortuosity ($\tau$)")
ax2.set_title("Best Found So Far")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 2. Comparing the Structures

How do the optimised structures differ from the random starting point? The slices below show the initial configuration alongside both optimised results. Look for the emergence of more connected, channel-like pore pathways in the optimised versions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), dpi=120)
mid = SIZE // 2

for ax, micro, title in zip(axes,
                             [initial_micro, greedy_micro, sa_micro],
                             ["Initial (random)", "Greedy result", "SA result"]):
    ax.imshow(micro[mid, :, :], cmap="gray", interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle(f"Mid-plane slices (z = {mid}), porosity = {POROSITY:.2f}",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. How Close to the Theoretical Bound?

For a structure with porosity $\varepsilon$ and no obstacles, the theoretical minimum tortuosity is $\tau = 1$ (a straight channel). No rearrangement of voxels at fixed porosity can beat this. Below we quantify how far each method got from the starting point and from the bound, and verify that porosity was preserved exactly throughout.

In [ ]:
tau_initial = greedy_tau[0]  # both methods started from the same structure
tau_min = 1.0               # theoretical lower bound (straight channel)

print("=== Summary ===\n")
print(f"{'':22s}  {'Greedy':>10s}  {'SA':>10s}")
print(f"{'-'*46}")
print(f"{'Initial tau':22s}  {tau_initial:10.4f}  {tau_initial:10.4f}")
print(f"{'Final best tau':22s}  {greedy_best[-1]:10.4f}  {sa_best[-1]:10.4f}")
print(f"{'Reduction':22s}  {(1 - greedy_best[-1]/tau_initial)*100:9.1f}%  {(1 - sa_best[-1]/tau_initial)*100:9.1f}%")
print(f"{'Distance from bound':22s}  {greedy_best[-1] - tau_min:10.4f}  {sa_best[-1] - tau_min:10.4f}")
print(f"{'Accepted / total':22s}  {greedy_acc:>4d}/{N_ITERATIONS:<4d}    {sa_acc:>4d}/{N_ITERATIONS:<4d}")
print()

# Verify porosity conservation
for name, micro in [("Initial", initial_micro), ("Greedy", greedy_micro), ("SA", sa_micro)]:
    print(f"  {name:8s} porosity = {micro.mean():.6f}")

print(f"\nPorosity is conserved by construction: every swap exchanges one pore "
      f"voxel for one solid voxel.")

## Scaling Up: From Toy Demo to Production Optimisation

This tutorial is a **toy demonstration** — 100 iterations on a $24^3$ grid with single-voxel swaps. But the same pattern scales directly to production runs on HPC, where OpenImpala's throughput transforms what is feasible:

### Scaling the inner loop (faster evaluations)

Launch the same script with MPI to speed up each PDE solve:
```bash
mpirun -np 64 python topology_opt.py --size 128 --iterations 5000
```

### Scaling the outer loop (more parallel searches)

Run many independent SA trajectories simultaneously via SLURM job arrays, then pick the best:
```bash
#SBATCH --array=0-31  # 32 independent SA runs with different random seeds
mpirun -np 16 python topology_opt.py --seed $SLURM_ARRAY_TASK_ID
```

This "multi-start" strategy is embarrassingly parallel and particularly effective for SA, where different random seeds explore different regions of configuration space.

### Production-grade improvements

- **Increase iterations** (1,000-10,000+) and domain size ($64^3$-$128^3$) for meaningful results.
- **Use smarter moves** — swap clusters of voxels, or use gradient-based sensitivity information to guide which voxels to swap.
- **Tune SA parameters** — the initial temperature and cooling schedule strongly affect performance. Adaptive schemes (e.g., reheating when the acceptance rate drops too low) are common in practice.
- **Optimise in multiple directions** — minimise the maximum tortuosity across X, Y, and Z for isotropic transport.
- **Add constraints** — in battery electrode design, for example, you may need to maintain mechanical connectivity of the solid phase while optimising ionic transport through the pore phase.
- **Combine with surrogates** — use the surrogate from [Tutorial 5](05_surrogate_modelling.ipynb) to pre-screen candidates cheaply, then validate the best ones with full OpenImpala solves.

**Continue the series:**
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Train a fast surrogate to replace the PDE solve, enabling much larger optimisation campaigns.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Deploy production-scale optimisation campaigns on a cluster.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **Topology optimisation for transport:** J. K. Guest & J. H. Prevost, *Topology optimization of creeping fluid flows using a Darcy-Stokes finite element*, Int. J. Numer. Methods Eng. 66(3), 461-484 (2006). [doi:10.1002/nme.1560](https://doi.org/10.1002/nme.1560)
3. **Simulated annealing:** S. Kirkpatrick, C. D. Gelatt Jr., & M. P. Vecchi, *Optimization by simulated annealing*, Science 220(4598), 671-680 (1983). [doi:10.1126/science.220.4598.671](https://doi.org/10.1126/science.220.4598.671)
4. **Electrode microstructure design:** G. M. Goldin et al., *Three-dimensional particle-resolved models of Li-ion batteries to assist the evaluation of empirical parameters in one-dimensional models*, Electrochimica Acta 64, 118-129 (2012). [doi:10.1016/j.electacta.2011.12.119](https://doi.org/10.1016/j.electacta.2011.12.119)
5. **Voxel-based topology optimisation:** O. Sigmund, *A 99 line topology optimization code written in Matlab*, Structural and Multidisciplinary Optimization 21(2), 120-127 (2001). [doi:10.1007/s001580050176](https://doi.org/10.1007/s001580050176)
6. **Hashin-Shtrikman bounds:** Z. Hashin & S. Shtrikman, *A variational approach to the theory of the effective magnetic permeability of multiphase materials*, J. Applied Physics 33(10), 3125-3131 (1962). [doi:10.1063/1.1728579](https://doi.org/10.1063/1.1728579)